**Preparação dos Dados**

In [22]:
import requests

url_dados = "https://huggingface.co/datasets/alexvaroz/nyc_taxi_trip_2024_p1_sample/resolve/main/nyc_tripdata_2024_sample_1M.csv"

resposta = requests.get(url_dados)


In [23]:
arquivo_base = open("nyc_taxi.csv", "w", encoding="utf-8")
arquivo_base.write(resposta.text)
arquivo_base.close()

print("Arquivo baixado")

Arquivo baixado


In [24]:
arquivo = open("nyc_taxi.csv", "r", encoding="utf-8")

cabecalho = arquivo.readline().strip()

arquivo.close()

print(cabecalho)

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,arquivo_origem


In [25]:
from collections import defaultdict
from functools import reduce

In [26]:
def shuffle(nome_arquivo_entrada, nome_arquivo_saida):
    resultado_intermediario = defaultdict(list)

    with open(nome_arquivo_entrada, "r", encoding="utf-8") as arquivo_entrada:
        for linha in arquivo_entrada:
            chave, valor = linha.strip().split("\t")
            resultado_intermediario[chave].append(float(valor))

    resultado = dict(resultado_intermediario)

    with open(nome_arquivo_saida, "w", encoding="utf-8") as arquivo_shuffle:
        arquivo_shuffle.write(str(resultado))

In [27]:
def somar_elementos(x, y):
    return x + y


def maior_valor(x, y):
    if x >= y:
        return x
    else:
        return y


def menor_valor(x, y):
    if x <= y:
        return x
    else:
        return y

**1. Número de viagens por tipo de pagamento.**

In [28]:
# Criando o Mapper
def map_viagens_por_pagamento(nome_arquivo_entrada, nome_arquivo_saida):
    arquivo_entrada = open(nome_arquivo_entrada, "r", encoding="utf-8")
    arquivo_saida = open(nome_arquivo_saida, "w", encoding="utf-8")

    primeira_linha = arquivo_entrada.readline()  # elimina o cabeçalho

    for linha in arquivo_entrada:
        dados = linha.strip().split(",")

        payment_type = dados[9]

        arquivo_saida.write("%s\t%s\n" % (payment_type, 1))

    arquivo_entrada.close()
    arquivo_saida.close()

In [29]:
# #Executando o Mapper
map_viagens_por_pagamento(
    "nyc_taxi.csv",
    "saida_mapper_pagamento.txt"
)

In [30]:
# Executando o Shuffer
shuffle(
    "saida_mapper_pagamento.txt",
    "saida_shuffle_pagamento.txt"
)

In [31]:
# Executando o Reducer
def reducer(nome_arquivo_entrada, nome_arquivo_saida, funcao_reduce):
    arquivo = open(nome_arquivo_entrada, "r", encoding="utf-8")
    conteudo_entrada = arquivo.read()
    dict_entrada = eval(conteudo_entrada)
    arquivo.close()

    dict_result = {}

    for chave, lista in dict_entrada.items():
        dict_result[chave] = reduce(funcao_reduce, lista)

    arquivo_reduce = open(nome_arquivo_saida, "w", encoding="utf-8")

    for item in sorted(dict_result, key=dict_result.get, reverse=True):
        arquivo_reduce.write(item + "\t" + str(int(dict_result[item])) + "\n")

    arquivo_reduce.close()

In [32]:
reducer(
    "saida_shuffle_pagamento.txt",
    "saida_reduce_pagamento.txt",
    somar_elementos
)

In [33]:
with open("saida_reduce_pagamento.txt", "r", encoding="utf-8") as arquivo:
    print(arquivo.read())

1	743405
2	136221
0	97124
4	16543
3	6707



In [39]:
tipos_pagamento = {
    "0": "Não especificado",
    "1": "Cartão de crédito",
    "2": "Dinheiro",
    "3": "Sem cobrança",
    "4": "Disputa"
}

In [40]:
print("Número de viagens por tipo de pagamento")
print("-" * 45)

with open("saida_reduce_pagamento.txt", "r", encoding="utf-8") as arquivo:
    for linha in arquivo:
        codigo, quantidade = linha.strip().split("\t")
        tipo = tipos_pagamento.get(codigo, "Desconhecido")
        print(f"{tipo}: {int(quantidade):,}".replace(",", "."))

Número de viagens por tipo de pagamento
---------------------------------------------
Cartão de crédito: 743.405
Dinheiro: 136.221
Não especificado: 97.124
Disputa: 16.543
Sem cobrança: 6.707


**2. Receita total por tipo de pagamento**

In [34]:
# Criando o Mapper
def map_receita_por_pagamento(nome_arquivo_entrada, nome_arquivo_saida):
    arquivo_entrada = open(nome_arquivo_entrada, "r", encoding="utf-8")
    arquivo_saida = open(nome_arquivo_saida, "w", encoding="utf-8")

    primeira_linha = arquivo_entrada.readline()  # elimina o cabeçalho

    for linha in arquivo_entrada:
        dados = linha.strip().split(",")

        payment_type = dados[9]
        fare_amount = float(dados[10])

        arquivo_saida.write("%s\t%s\n" % (payment_type, fare_amount))

    arquivo_entrada.close()
    arquivo_saida.close()

In [35]:
# Executando o Mapper
map_receita_por_pagamento(
    "nyc_taxi.csv",
    "saida_mapper_receita.txt"
)

In [36]:
# Shuffle
shuffle(
    "saida_mapper_receita.txt",
    "saida_shuffle_receita.txt"
)

In [37]:
reducer(
    "saida_shuffle_receita.txt",
    "saida_reduce_receita.txt",
    somar_elementos
)

In [38]:
with open("saida_reduce_receita.txt", "r", encoding="utf-8") as arquivo:
    print(arquivo.read())

1	14407655
2	2483767
0	1910472
3	40398
4	18002



In [41]:
print("Receita total por tipo de pagamento")
print("-" * 45)

with open("saida_reduce_receita.txt", "r", encoding="utf-8") as arquivo:
    for linha in arquivo:
        codigo, receita = linha.strip().split("\t")
        tipo = tipos_pagamento.get(codigo, "Desconhecido")
        receita = float(receita)
        print(f"{tipo}: US$ {receita:,.2f}".replace(",", "X").replace(".", ",").replace("X", "."))

Receita total por tipo de pagamento
---------------------------------------------
Cartão de crédito: US$ 14.407.655,00
Dinheiro: US$ 2.483.767,00
Não especificado: US$ 1.910.472,00
Sem cobrança: US$ 40.398,00
Disputa: US$ 18.002,00


**3. Tarifa média cobrada nas viagens.**

In [42]:
# Mapper
def map_tarifa_media(nome_arquivo_entrada, nome_arquivo_saida):
    arquivo_entrada = open(nome_arquivo_entrada, "r", encoding="utf-8")
    arquivo_saida = open(nome_arquivo_saida, "w", encoding="utf-8")

    primeira_linha = arquivo_entrada.readline()

    for linha in arquivo_entrada:
        dados = linha.strip().split(",")

        fare_amount = float(dados[10])

        arquivo_saida.write("soma\t%s\n" % fare_amount)
        arquivo_saida.write("quantidade\t1\n")

    arquivo_entrada.close()
    arquivo_saida.close()

In [43]:
# Executando o Mapper
map_tarifa_media(
    "nyc_taxi.csv",
    "saida_mapper_tarifa_media.txt"
)

In [44]:
shuffle(
    "saida_mapper_tarifa_media.txt",
    "saida_shuffle_tarifa_media.txt"
)

In [45]:
# Reducer
reducer(
    "saida_shuffle_tarifa_media.txt",
    "saida_reduce_tarifa_media.txt",
    somar_elementos
)

In [46]:
with open("saida_reduce_tarifa_media.txt", "r", encoding="utf-8") as arquivo:
    print(arquivo.read())

soma	18860296
quantidade	1000000



In [48]:
resultado = {}

with open("saida_reduce_tarifa_media.txt", "r", encoding="utf-8") as arquivo:
    for linha in arquivo:
        chave, valor = linha.strip().split("\t")
        resultado[chave] = float(valor)

tarifa_media = resultado["soma"] / resultado["quantidade"]


In [49]:
print(f"Tarifa média das viagens: US$ {tarifa_media:.2f}")

Tarifa média das viagens: US$ 18.86


4. Data e hora em que foi feita a viagem mais longa

In [50]:
# Mapper
def map_viagem_mais_longa(nome_arquivo_entrada, nome_arquivo_saida):
    arquivo_entrada = open(nome_arquivo_entrada, "r", encoding="utf-8")
    arquivo_saida = open(nome_arquivo_saida, "w", encoding="utf-8")

    primeira_linha = arquivo_entrada.readline()

    for linha in arquivo_entrada:
        dados = linha.strip().split(",")

        pickup_datetime = dados[1]
        trip_distance = float(dados[4])

        arquivo_saida.write(
            "%s\t%s\n" % (pickup_datetime, trip_distance)
        )

    arquivo_entrada.close()
    arquivo_saida.close()

In [51]:
map_viagem_mais_longa(
    "nyc_taxi.csv",
    "saida_mapper_viagem_mais_longa.txt"
)

In [53]:
shuffle(
    "saida_mapper_viagem_mais_longa.txt",
    "saida_shuffle_viagem_mais_longa.txt"
)

In [54]:
# Reducer
def reducer_viagem_mais_longa(nome_arquivo_entrada, nome_arquivo_saida):
    arquivo = open(nome_arquivo_entrada, "r", encoding="utf-8")

    conteudo = arquivo.read()
    dados = eval(conteudo)

    arquivo.close()

    maior_distancia = -1
    data_hora = ""

    for chave, lista in dados.items():
        distancia = max(lista)

        if distancia > maior_distancia:
            maior_distancia = distancia
            data_hora = chave

    arquivo_saida = open(nome_arquivo_saida, "w", encoding="utf-8")

    arquivo_saida.write(
        "%s\t%s\n" % (data_hora, maior_distancia)
    )

    arquivo_saida.close()

In [55]:
reducer_viagem_mais_longa(
    "saida_shuffle_viagem_mais_longa.txt",
    "saida_reduce_viagem_mais_longa.txt"
)

In [58]:
with open("saida_reduce_viagem_mais_longa.txt", "r", encoding="utf-8") as arquivo:
    data_hora, distancia = arquivo.readline().strip().split("\t")

print("Viagem mais longa")
print("-" * 44)
print(f"Data e hora de início: {data_hora}")
print(f"Distância percorrida: {float(distancia):.2f} milhas")

Viagem mais longa
--------------------------------------------
Data e hora de início: 2024-05-10 17:33:00
Distância percorrida: 86789.20 milhas


**5. Quantidade de viagens por hora.**

In [59]:
def map_viagens_por_hora(nome_arquivo_entrada, nome_arquivo_saida):
    arquivo_entrada = open(nome_arquivo_entrada, "r", encoding="utf-8")
    arquivo_saida = open(nome_arquivo_saida, "w", encoding="utf-8")

    primeira_linha = arquivo_entrada.readline()

    for linha in arquivo_entrada:
        dados = linha.strip().split(",")

        pickup_datetime = dados[1]

        # A hora está nas posições 11 e 12 da data/hora
        hora = pickup_datetime[11:13]

        arquivo_saida.write("%s\t%s\n" % (hora, 1))

    arquivo_entrada.close()
    arquivo_saida.close()

In [60]:
map_viagens_por_hora(
    "nyc_taxi.csv",
    "saida_mapper_viagens_por_hora.txt"
)

In [61]:
shuffle(
    "saida_mapper_viagens_por_hora.txt",
    "saida_shuffle_viagens_por_hora.txt"
)

In [62]:
reducer(
    "saida_shuffle_viagens_por_hora.txt",
    "saida_reduce_viagens_por_hora.txt",
    somar_elementos
)

In [64]:
resultado_horas = {}

with open("saida_reduce_viagens_por_hora.txt", "r", encoding="utf-8") as arquivo:
    for linha in arquivo:
        hora, quantidade = linha.strip().split("\t")
        resultado_horas[hora] = int(quantidade)

print("Quantidade de viagens por hora")
print("-" * 33)

for hora in sorted(resultado_horas):
    print(f"{hora}:00 — {resultado_horas[hora]:,}".replace(",", "."))

Quantidade de viagens por hora
---------------------------------
00:00 — 29.165
01:00 — 18.822
02:00 — 12.280
03:00 — 8.281
04:00 — 6.054
05:00 — 6.194
06:00 — 13.966
07:00 — 28.065
08:00 — 38.308
09:00 — 42.309
10:00 — 44.804
11:00 — 48.295
12:00 — 53.128
13:00 — 55.362
14:00 — 59.345
15:00 — 60.205
16:00 — 61.563
17:00 — 67.880
18:00 — 71.403
19:00 — 62.752
20:00 — 56.542
21:00 — 58.333
22:00 — 54.581
23:00 — 42.363


**6. Distância total percorrida por hora.**

In [65]:
# Mapper
def map_distancia_por_hora(nome_arquivo_entrada, nome_arquivo_saida):
    arquivo_entrada = open(nome_arquivo_entrada, "r", encoding="utf-8")
    arquivo_saida = open(nome_arquivo_saida, "w", encoding="utf-8")

    primeira_linha = arquivo_entrada.readline()

    for linha in arquivo_entrada:
        dados = linha.strip().split(",")

        pickup_datetime = dados[1]
        trip_distance = float(dados[4])

        hora = pickup_datetime[11:13]

        arquivo_saida.write("%s\t%s\n" % (hora, trip_distance))

    arquivo_entrada.close()
    arquivo_saida.close()

In [66]:
map_distancia_por_hora(
    "nyc_taxi.csv",
    "saida_mapper_distancia_por_hora.txt"
)

In [67]:
shuffle(
    "saida_mapper_distancia_por_hora.txt",
    "saida_shuffle_distancia_por_hora.txt"
)

In [68]:
reducer(
    "saida_shuffle_distancia_por_hora.txt",
    "saida_reduce_distancia_por_hora.txt",
    somar_elementos
)

In [71]:
resultado_distancias = {}

with open("saida_reduce_distancia_por_hora.txt", "r", encoding="utf-8") as arquivo:
    for linha in arquivo:
        hora, distancia = linha.strip().split("\t")
        resultado_distancias[hora] = float(distancia)

print("Distância total percorrida por hora")
print("-" * 37)

for hora in sorted(resultado_distancias):
    distancia = resultado_distancias[hora]
    print(f"{hora}:00 — {distancia:,.2f} milhas".replace(",", "X").replace(".", ",").replace("X", "."))

Distância total percorrida por hora
-------------------------------------
00:00 — 109.568,00 milhas
01:00 — 60.695,00 milhas
02:00 — 36.643,00 milhas
03:00 — 29.745,00 milhas
04:00 — 28.350,00 milhas
05:00 — 91.136,00 milhas
06:00 — 131.056,00 milhas
07:00 — 195.237,00 milhas
08:00 — 169.111,00 milhas
09:00 — 222.652,00 milhas
10:00 — 138.342,00 milhas
11:00 — 145.189,00 milhas
12:00 — 166.186,00 milhas
13:00 — 190.847,00 milhas
14:00 — 215.732,00 milhas
15:00 — 312.374,00 milhas
16:00 — 238.718,00 milhas
17:00 — 321.659,00 milhas
18:00 — 252.601,00 milhas
19:00 — 265.021,00 milhas
20:00 — 267.461,00 milhas
21:00 — 283.775,00 milhas
22:00 — 189.960,00 milhas
23:00 — 161.753,00 milhas
